# 5. The test suite

The repository's tests, run in this kernel. In the browser the test dependencies are installed at the versions recorded in `uv.lock`, read from the file itself. The cell follows the lock from the suite's direct dependencies to every package that applies on this platform, and installs exactly that set with `deps=False`, so micropip resolves nothing and the result does not depend on the order in which it fetches. pydantic is the exception: its compiled core has to match the Pyodide build, so it comes from the Pyodide distribution, with its own dependencies, in a second call made once the pinned set is in place.

Three markers name a capability a WebAssembly runtime may lack, and `tests/conftest.py` skips a marked test where the capability is missing:

| Marker | Tests | Skipped where |
| --- | ---: | --- |
| `threads` | 18 | the runtime cannot start a thread: Pyodide |
| `subprocess` | 1 | the runtime has no processes: Pyodide |
| `event_loop` | 9 | synchronous code cannot run an event loop: Pyodide in an engine without JSPI, and a desktop notebook, whose kernel is already running one |

Chromium and Firefox both support JSPI, so in either browser the 19 `threads` and `subprocess` tests are skipped and the `event_loop` tests run. An engine without JSPI skips all 28. On the desktop this notebook skips the 9 `event_loop` tests, and `make test` runs them outside a notebook. Notebook 3 runs the asyncio scenarios with top-level `await`, which needs neither threads nor JSPI.

In [ ]:
import platform
import re
import sys
import tomllib
from importlib import metadata
from pathlib import Path

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'pytest.ini').exists())
lock = tomllib.loads((root / 'uv.lock').read_text())['package']
version = {package['name']: package['version'] for package in lock}
requires = {package['name']: package.get('dependencies', []) for package in lock}
here = {'sys_platform': sys.platform, 'platform_machine': platform.machine()}


def applies(marker):
    """Evaluate a marker of the form `name == 'value' or ...`, the only form this closure contains."""
    if not marker:
        return True
    clauses = [re.fullmatch(r"(\w+) == '([^']*)'", clause) for clause in marker.split(' or ')]
    if not all(clause and clause[1] in here for clause in clauses):
        raise ValueError(f'marker not understood: {marker}')
    return any(here[clause[1]] == clause[2] for clause in clauses)


closure, pending = set(), ['pytest', 'pytest-bdd', 'factory-boy', 'sqlalchemy', 'attrs']
while pending:
    name = pending.pop()
    if name not in closure:
        closure.add(name)
        pending += [dependency['name'] for dependency in requires[name] if applies(dependency.get('marker', ''))]
pinned = sorted(f'{name}=={version[name]}' for name in closure)

if sys.platform == 'emscripten':  # in the browser: this repository's wheel, served with the site
    import piplite

    await piplite.install(pinned, deps=False)
    await piplite.install(['banking', 'pydantic'])

print('Python', sys.version.split()[0], 'on', sys.platform)
print(f'pinned from uv.lock ({len(pinned)}):', ', '.join(pinned))

# pydantic's own requirements, read from its metadata, with the version each one arrived at.
needs = [
    re.match(r'[\w.-]+', requirement)[0]
    for requirement in metadata.requires('pydantic')
    if 'extra ==' not in requirement
]
print(
    'pydantic',
    metadata.version('pydantic'),
    'requires:',
    ', '.join(f'{name} {metadata.version(name)}' for name in needs),
)

In [ ]:
import os

import pytest

sys.dont_write_bytecode = True  # keeps __pycache__ out of the notebook's file browser
os.chdir(root)
exit_code = pytest.main(['tests', '-p', 'no:cacheprovider', '-q', '-rs'])
print('pytest exit code:', int(exit_code))